# Luma Analytics▲ — Nonprofit Donor Intelligence
### University Donor Dataset · Random Forest Churn + LTV Model
**Replace the CSV path in Cell 2 with your client's file. Everything else runs automatically.**

In [ ]:
# CELL 1 — Install dependencies
!pip install pandas numpy scikit-learn matplotlib seaborn sqlalchemy psycopg2-binary --quiet

In [ ]:
# CELL 2 — CONFIG: change these for each client
CSV_PATH = 'data_science_for_fundraising_donor_data.csv'  # <-- swap with client CSV

# Supabase connection (get from Supabase → Settings → Database → Connection string)
SUPABASE_URL = 'postgresql://postgres:[PASSWORD]@[HOST]:5432/postgres'

# Table name in Supabase (change per client)
TABLE_NAME = 'university_donors'

In [ ]:
# CELL 3 — Load & inspect raw data
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(CSV_PATH)
print(f'Rows: {len(df):,} | Columns: {len(df.columns)}')
print(f'Columns: {list(df.columns)}')
df.head(3)

In [ ]:
# CELL 4 — Clean data

# 1. Strip $ and commas from giving columns, convert to float
giving_cols = ['PrevFYGiving','PrevFY1Giving','PrevFY2Giving','PrevFY3Giving','PrevFY4Giving','CurrFYGiving']
for col in giving_cols:
    df[col] = df[col].astype(str).str.replace(r'[\$,]', '', regex=True).replace('nan','0').astype(float)

# 2. Clean binary Y/N columns to 1/0
binary_cols = ['MEMBERSHIP_IND','ALUMNUS_IND','PARENT_IND','HAS_INVOLVEMENT_IND','EMAIL_PRESENT_IND']
for col in binary_cols:
    df[col] = (df[col].str.strip().str.upper() == 'Y').astype(int)

# 3. Define CHURN: gave last year but not this year = lapsed
df['churned'] = ((df['PrevFYGiving'] > 0) & (df['CurrFYGiving'] == 0)).astype(int)

# 4. Define IS_DONOR: has ever given
df['is_donor'] = (df['DONOR_IND'].str.strip().str.upper() == 'Y').astype(int)

# 5. Total giving across 4 previous years
df['total_prev_giving'] = df[['PrevFYGiving','PrevFY1Giving','PrevFY2Giving','PrevFY3Giving','PrevFY4Giving']].sum(axis=1)

# 6. Giving consistency score (how many of 5 years did they give)
df['giving_years'] = (df[giving_cols[:-1]] > 0).sum(axis=1)

# 7. Year over year giving trend (positive = growing, negative = declining)
df['giving_trend'] = df['PrevFYGiving'] - df['PrevFY1Giving']

# 8. Fill missing age with median
df['AGE'] = df['AGE'].fillna(df['AGE'].median())

# 9. Encode gender
df['gender_enc'] = (df['GENDER'].str.strip().str.upper() == 'M').astype(int)

# 10. Clean wealth rating — extract midpoint
def parse_wealth(w):
    if pd.isna(w): return 0
    w = str(w).replace('$','').replace(',','')
    parts = w.split('-')
    try:
        nums = [float(p.strip()) for p in parts if p.strip().replace('.','').isdigit()]
        return np.mean(nums) if nums else 0
    except: return 0

df['wealth_numeric'] = df['WEALTH_RATING'].apply(parse_wealth)

print(f'Clean dataset shape: {df.shape}')
print(f'Lapsed donors (churned): {df["churned"].sum():,}')
print(f'Active donors: {(df["CurrFYGiving"]>0).sum():,}')
print(f'Never donated: {(df["total_prev_giving"]==0).sum():,}')
df[['ID','AGE','PrevFYGiving','CurrFYGiving','churned','giving_years','giving_trend']].head(5)

In [ ]:
# CELL 5 — Exploratory Analysis
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Luma Analytics▲ — Donor Data Overview', fontsize=16, fontweight='bold', y=1.02)

# 1. Giving distribution (non-zero donors only)
donors_only = df[df['total_prev_giving'] > 0]
axes[0,0].hist(np.log1p(donors_only['total_prev_giving']), bins=40, color='#0D1117', edgecolor='#F5A623', linewidth=0.5)
axes[0,0].set_title('Total Giving Distribution (log scale)', fontweight='bold')
axes[0,0].set_xlabel('Log(Total Giving)')
axes[0,0].set_ylabel('# Donors')

# 2. Churn breakdown
prev_donors = df[df['PrevFYGiving'] > 0]
churn_counts = prev_donors['churned'].value_counts()
axes[0,1].pie(churn_counts, labels=['Lapsed','Retained'], colors=['#E84545','#22C55E'],
              autopct='%1.1f%%', startangle=90, textprops={'fontsize':11})
axes[0,1].set_title('Donor Retention (Prior Year Givers)', fontweight='bold')

# 3. Giving by alumni status
alum_giving = df.groupby('ALUMNUS_IND')['total_prev_giving'].mean()
axes[0,2].bar(['Non-Alumni','Alumni'], alum_giving.values, color=['#0D1117','#F5A623'], edgecolor='none')
axes[0,2].set_title('Avg Giving: Alumni vs Non-Alumni', fontweight='bold')
axes[0,2].set_ylabel('Avg Total Giving ($)')

# 4. Giving years consistency
axes[1,0].hist(df['giving_years'], bins=6, color='#F5A623', edgecolor='#0D1117', linewidth=0.5)
axes[1,0].set_title('Giving Consistency (# of 5 Years Donated)', fontweight='bold')
axes[1,0].set_xlabel('Years with Donation')

# 5. Email presence vs giving
email_giving = df.groupby('EMAIL_PRESENT_IND')['total_prev_giving'].mean()
axes[1,1].bar(['No Email','Has Email'], email_giving.values, color=['#0D1117','#F5A623'])
axes[1,1].set_title('Avg Giving: Email on File?', fontweight='bold')
axes[1,1].set_ylabel('Avg Total Giving ($)')

# 6. YoY giving trend
axes[1,2].hist(df[df['giving_trend']!=0]['giving_trend'].clip(-1000,1000), bins=40, color='#0D1117', edgecolor='#F5A623', linewidth=0.5)
axes[1,2].axvline(0, color='#E84545', linewidth=2, linestyle='--', label='No change')
axes[1,2].set_title('YoY Giving Trend (Prev2 → Prev1 FY)', fontweight='bold')
axes[1,2].set_xlabel('Change in Giving ($)')
axes[1,2].legend()

plt.tight_layout()
plt.savefig('donor_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA charts saved as donor_eda.png')

In [ ]:
# CELL 6 — Churn Prediction Model (Random Forest Classifier)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# Only model on donors who gave last year (churn is only meaningful for these)
model_df = df[df['PrevFYGiving'] > 0].copy()
print(f'Training on {len(model_df):,} donors who gave last year')
print(f'Churn rate: {model_df["churned"].mean()*100:.1f}%')

# Features for churn model
FEATURES = [
    'PrevFYGiving',        # Last year gift amount
    'PrevFY1Giving',       # 2 years ago
    'PrevFY2Giving',       # 3 years ago
    'total_prev_giving',   # Total historical giving
    'giving_years',        # Giving consistency
    'giving_trend',        # Growing or declining
    'CON_YEARS',           # Years as constituent
    'AGE',                 # Age
    'ALUMNUS_IND',         # Alumni status
    'MEMBERSHIP_IND',      # Member
    'PARENT_IND',          # Parent
    'HAS_INVOLVEMENT_IND', # Campus involvement
    'EMAIL_PRESENT_IND',   # Email on file
    'wealth_numeric',      # Estimated wealth
    'gender_enc',          # Gender
]

X = model_df[FEATURES].fillna(0)
y = model_df['churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train Random Forest
clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    class_weight='balanced',  # handles class imbalance
    random_state=42,
    n_jobs=-1
)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_prob)

print(f'\nModel Performance:')
print(f'AUC-ROC: {auc:.3f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Retained','Churned']))

In [ ]:
# CELL 7 — Feature Importance (Why donors churn)
import matplotlib.pyplot as plt

feat_imp = pd.DataFrame({
    'feature': FEATURES,
    'importance': clf.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(feat_imp['feature'], feat_imp['importance'], color='#F5A623', edgecolor='#0D1117', linewidth=0.5)
ax.set_title('Luma Analytics▲ — What Drives Donor Churn\nRandom Forest Feature Importance', 
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Importance Score', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add value labels
for bar, val in zip(bars, feat_imp['importance']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, 
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 churn drivers:')
print(feat_imp.tail(5)[['feature','importance']].to_string())

In [ ]:
# CELL 8 — Score ALL donors with churn probability

# Score the donors who gave last year
X_all = model_df[FEATURES].fillna(0)
model_df = model_df.copy()
model_df['churn_probability'] = clf.predict_proba(X_all)[:,1].round(3)
model_df['risk_tier'] = pd.cut(
    model_df['churn_probability'],
    bins=[0, 0.35, 0.65, 1.0],
    labels=['Low', 'Medium', 'High']
)

print('Risk tier breakdown:')
print(model_df['risk_tier'].value_counts())
print(f'\nRevenue at risk (High risk donors, prev year giving):')
print(f"${model_df[model_df['risk_tier']=='High']['PrevFYGiving'].sum():,.0f}")

print('\nTop 10 highest value donors at risk:')
top_at_risk = model_df[model_df['risk_tier']=='High'].nlargest(10, 'PrevFYGiving')[
    ['ID','PrevFYGiving','total_prev_giving','churn_probability','risk_tier','ALUMNUS_IND','EMAIL_PRESENT_IND']
]
print(top_at_risk.to_string())

In [ ]:
# CELL 9 — LTV Prediction Model (Random Forest Regressor)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Train LTV model on all donors who have ever given
ltv_df = df[df['TotalGiving'] > 0].copy()
print(f'Training LTV model on {len(ltv_df):,} donors who have given')

LTV_FEATURES = [
    'PrevFYGiving', 'PrevFY1Giving', 'PrevFY2Giving', 'PrevFY3Giving', 'PrevFY4Giving',
    'giving_years', 'giving_trend', 'CON_YEARS', 'AGE',
    'ALUMNUS_IND', 'MEMBERSHIP_IND', 'PARENT_IND', 'HAS_INVOLVEMENT_IND',
    'EMAIL_PRESENT_IND', 'wealth_numeric', 'gender_enc'
]

X_ltv = ltv_df[LTV_FEATURES].fillna(0)
y_ltv = ltv_df['TotalGiving']

X_ltv_train, X_ltv_test, y_ltv_train, y_ltv_test = train_test_split(X_ltv, y_ltv, test_size=0.2, random_state=42)

reg = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
reg.fit(X_ltv_train, y_ltv_train)

y_ltv_pred = reg.predict(X_ltv_test)
mae = mean_absolute_error(y_ltv_test, y_ltv_pred)
r2 = r2_score(y_ltv_test, y_ltv_pred)

print(f'LTV Model Performance:')
print(f'MAE: ${mae:,.0f}')
print(f'R²: {r2:.3f}')

# Score all donors
X_all_ltv = df[LTV_FEATURES].fillna(0)
df['predicted_ltv'] = reg.predict(X_all_ltv).round(2)
df['predicted_ltv'] = df['predicted_ltv'].clip(lower=0)

print(f'\nAvg predicted LTV: ${df["predicted_ltv"].mean():,.0f}')
print(f'Total predicted future value: ${df["predicted_ltv"].sum():,.0f}')

In [ ]:
# CELL 10 — Next Year Giving Prediction

# Predict CurrFYGiving using prior years as features
giving_df = df[df['PrevFYGiving'] > 0].copy()

NYG_FEATURES = [
    'PrevFYGiving','PrevFY1Giving','PrevFY2Giving','PrevFY3Giving','PrevFY4Giving',
    'giving_years','giving_trend','CON_YEARS','AGE',
    'ALUMNUS_IND','MEMBERSHIP_IND','HAS_INVOLVEMENT_IND','EMAIL_PRESENT_IND','wealth_numeric'
]

X_nyg = giving_df[NYG_FEATURES].fillna(0)
y_nyg = giving_df['CurrFYGiving']

X_nyg_tr, X_nyg_te, y_nyg_tr, y_nyg_te = train_test_split(X_nyg, y_nyg, test_size=0.2, random_state=42)

reg_nyg = RandomForestRegressor(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1)
reg_nyg.fit(X_nyg_tr, y_nyg_tr)

giving_df['predicted_next_gift'] = reg_nyg.predict(X_nyg).clip(0).round(2)

mae_nyg = mean_absolute_error(y_nyg_te, reg_nyg.predict(X_nyg_te))
print(f'Next Gift Prediction MAE: ${mae_nyg:,.0f}')
print(f'Total predicted revenue next year (active donors): ${giving_df["predicted_next_gift"].sum():,.0f}')

# Merge back
df = df.merge(
    giving_df[['ID','churn_probability','risk_tier','predicted_next_gift']].rename(
        columns={'churn_probability':'churn_prob_temp','risk_tier':'risk_tier_temp','predicted_next_gift':'predicted_next_gift_temp'}
    ), on='ID', how='left'
)
df['churn_probability'] = df['churn_prob_temp'].fillna(0)
df['risk_tier'] = df['risk_tier_temp'].fillna('Not Applicable')
df['predicted_next_gift'] = df['predicted_next_gift_temp'].fillna(0)
df.drop(columns=['churn_prob_temp','risk_tier_temp','predicted_next_gift_temp'], inplace=True)

print('\nFinal dataset columns:')
print([c for c in df.columns])

In [ ]:
# CELL 11 — Build final output table for Supabase

output = df[[
    'ID', 'AGE', 'GENDER', 'MARITAL_STATUS', 'ZIPCODE',
    'ALUMNUS_IND', 'MEMBERSHIP_IND', 'PARENT_IND', 'HAS_INVOLVEMENT_IND',
    'EMAIL_PRESENT_IND', 'CON_YEARS',
    'PrevFYGiving', 'PrevFY1Giving', 'PrevFY2Giving', 'PrevFY3Giving', 'PrevFY4Giving',
    'CurrFYGiving', 'TotalGiving',
    'total_prev_giving', 'giving_years', 'giving_trend',
    'wealth_numeric', 'is_donor', 'churned',
    # ML OUTPUTS — these are new columns Metabase will use
    'churn_probability',
    'risk_tier',
    'predicted_ltv',
    'predicted_next_gift',
]].copy()

# Add recommended action based on risk
def get_action(row):
    if row['risk_tier'] == 'High' and row['PrevFYGiving'] >= 500:
        return 'Personal Call — Major Donor'
    elif row['risk_tier'] == 'High':
        return 'Re-engagement Email'
    elif row['risk_tier'] == 'Medium':
        return 'Stewardship Newsletter'
    elif row['risk_tier'] == 'Low':
        return 'Upgrade Ask'
    else:
        return 'Cultivation'

output['recommended_action'] = output.apply(get_action, axis=1)

# Save to CSV
output.to_csv('donor_scores_for_supabase.csv', index=False)
print(f'Output shape: {output.shape}')
print(f'\nSample output (top 5 high risk donors):')
print(output[output['risk_tier']=='High'].nlargest(5,'PrevFYGiving')[[
    'ID','PrevFYGiving','churn_probability','risk_tier','predicted_ltv','predicted_next_gift','recommended_action'
]].to_string())

In [ ]:
# CELL 12 — Load into Supabase (PostgreSQL)
# SKIP THIS CELL if you just want the CSV — only run if Supabase is set up

from sqlalchemy import create_engine

engine = create_engine(SUPABASE_URL)

output.to_sql(
    TABLE_NAME,
    engine,
    if_exists='replace',   # replace table if it already exists
    index=False,
    chunksize=1000
)

print(f'✓ {len(output):,} rows loaded into Supabase table: {TABLE_NAME}')
print('Now connect Metabase to Supabase and build your dashboard!')

In [ ]:
# CELL 13 — Summary stats to paste into your client email

prev_donors = output[output['PrevFYGiving'] > 0]
high_risk = output[output['risk_tier']=='High']

print('='*50)
print('LUMA ANALYTICS — CLIENT SUMMARY')
print('='*50)
print(f'Total records analyzed:     {len(output):,}')
print(f'Active donors (gave last yr): {(output["PrevFYGiving"]>0).sum():,}')
print(f'Churn rate (prev donors):   {prev_donors["churned"].mean()*100:.1f}%')
print(f'High risk donors:           {len(high_risk):,}')
print(f'Revenue at risk:            ${high_risk["PrevFYGiving"].sum():,.0f}')
print(f'Avg predicted LTV:          ${output["predicted_ltv"].mean():,.0f}')
print(f'Projected next yr revenue:  ${output["predicted_next_gift"].sum():,.0f}')
print('='*50)
print('Files saved:')
print('  donor_scores_for_supabase.csv — load this into Supabase')
print('  donor_eda.png — exploratory charts')
print('  feature_importance.png — what drives churn')